In [1]:
#importing dataset from site
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition = fetch_ucirepo(id=544) 
  
# data (as pandas dataframes) 
X = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.features 
y = estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.data.targets 
  
# metadata 
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.metadata) 
  
# variable information 
print(estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition.variables) 


{'uci_id': 544, 'name': 'Estimation of Obesity Levels Based On Eating Habits and Physical Condition ', 'repository_url': 'https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition', 'data_url': 'https://archive.ics.uci.edu/static/public/544/data.csv', 'abstract': 'This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition. ', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 2111, 'num_features': 16, 'feature_types': ['Integer'], 'demographics': ['Gender', 'Age'], 'target_col': ['NObeyesdad'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Tue Sep 10 2024', 'dataset_doi': '10.24432/C5H31Z', 'creators': [], 'intro_paper': {'ID': 358, 'type': 

## Preprocessing Data

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# work on a combined copy so features/target stay aligned through cleaning
df = pd.concat([X, y], axis=1)

# 1. Remove null/NA values
df = df.dropna()

# 2. Remove redundant (duplicate) rows
df = df.drop_duplicates()

# 3. Convert categorical variables to numerical variables
# binary yes/no columns -> 0/1
binary_map = {'yes': 1, 'no': 0}
df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})
df['family_history_with_overweight'] = df['family_history_with_overweight'].map(binary_map)
df['FAVC'] = df['FAVC'].map(binary_map)
df['SMOKE'] = df['SMOKE'].map(binary_map)
df['SCC'] = df['SCC'].map(binary_map)

# ordinal columns (there's a natural low -> high ordering to these answers)
frequency_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df['CAEC'] = df['CAEC'].map(frequency_map)
df['CALC'] = df['CALC'].map(frequency_map)

# MTRANS has no natural order, so one-hot encode it instead
df = pd.get_dummies(df, columns=['MTRANS'], prefix='MTRANS', drop_first=True)
mtrans_cols = [c for c in df.columns if c.startswith('MTRANS_')]
df[mtrans_cols] = df[mtrans_cols].astype(int)

# target is an ordinal obesity level, encode it 0 (lowest) -> 6 (highest) for regression
obesity_order = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6,
}
df['NObeyesdad'] = df['NObeyesdad'].map(obesity_order)

# Weight is dropped: NObeyesdad is defined directly from BMI (weight/height^2),
# so keeping Weight as a predictor would leak the target into the features
df = df.drop(columns=['Weight'])

# 4. Drop features that show little to no correlation with the target
correlations = df.corr(numeric_only=True)['NObeyesdad'].drop('NObeyesdad').sort_values()
print('Correlation with target (NObeyesdad):')
print(correlations)

low_corr_threshold = 0.05
low_corr_cols = correlations[correlations.abs() < low_corr_threshold].index.tolist()
print(f'\nDropping low-correlation columns: {low_corr_cols}')
df = df.drop(columns=low_corr_cols)

# 5. Split back into features/target and standardize the features
y_clean = df['NObeyesdad']
X_clean = df.drop(columns=['NObeyesdad'])

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_clean), columns=X_clean.columns, index=X_clean.index)

print(f'\nFinal preprocessed shape: {X_scaled.shape}')
X_scaled.head()